# 龙头低吸策略分析

这个 Notebook 用于分析和优化龙头低吸策略

## 目录

1. 环境设置
2. 数据探索
3. 市场情绪分析
4. 龙头识别
5. 策略回测
6. 结果分析
7. 参数优化

## 1. 环境设置

In [ ]:
import sys
from pathlib import Path

# 添加项目路径
project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']  # 中文显示
plt.rcParams['axes.unicode_minus'] = False

print("✓ 环境设置完成")

## 2. 数据探索

In [ ]:
from data.akshare_provider import AKShareProvider
from data.emotion_data import EmotionDataManager

# 初始化数据提供器
provider = AKShareProvider()
emotion_manager = EmotionDataManager()

# 获取股票列表
stock_list = provider.get_stock_list()
print(f"A股总数: {len(stock_list)}")
print(f"前10只: {stock_list[:10]}")

### 2.1 查看个股数据

In [ ]:
# 获取平安银行数据
symbol = '000001.SZ'
start_date = '20240101'
end_date = '20241231'

df = provider.get_stock_daily(symbol, start_date, end_date)

if df is not None:
    print(f"\n{symbol} 数据:")
    print(df.head())
    print(f"\n数据条数: {len(df)}")
    
    # 绘制K线图
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
    
    # 价格
    ax1.plot(df.index, df['close'], label='收盘价', linewidth=2)
    ax1.fill_between(df.index, df['low'], df['high'], alpha=0.3)
    ax1.set_ylabel('价格')
    ax1.set_title(f'{symbol} 价格走势')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 成交量
    ax2.bar(df.index, df['volume'], alpha=0.7)
    ax2.set_ylabel('成交量')
    ax2.set_xlabel('日期')
    ax2.set_title('成交量')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 3. 市场情绪分析

In [ ]:
# 获取市场情绪数据
emotion_df = emotion_manager.emotion_df

if not emotion_df.empty:
    print("市场情绪统计:")
    print(emotion_df.describe())
    
    # 绘制情绪指标
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 上涨家数
    axes[0, 0].plot(emotion_df.index, emotion_df['up_count'], color='red')
    axes[0, 0].axhline(y=1000, color='blue', linestyle='--', label='冰点阈值')
    axes[0, 0].axhline(y=3500, color='green', linestyle='--', label='高潮阈值')
    axes[0, 0].set_title('上涨家数')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 涨停家数
    axes[0, 1].plot(emotion_df.index, emotion_df['limit_up_count'], color='red')
    axes[0, 1].set_title('涨停家数')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 上涨家数分布
    axes[1, 0].hist(emotion_df['up_count'], bins=50, alpha=0.7)
    axes[1, 0].axvline(x=1000, color='blue', linestyle='--', label='冰点')
    axes[1, 0].axvline(x=3500, color='green', linestyle='--', label='高潮')
    axes[1, 0].set_title('上涨家数分布')
    axes[1, 0].set_xlabel('上涨家数')
    axes[1, 0].set_ylabel('频数')
    axes[1, 0].legend()
    
    # 市场总成交额
    axes[1, 1].plot(emotion_df.index, emotion_df['total_amount'] / 1e8, color='blue')
    axes[1, 1].set_title('市场总成交额（亿元）')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 统计冰点和高潮天数
    ice_days = (emotion_df['up_count'] < 1000).sum()
    climax_days = (emotion_df['up_count'] > 3500).sum()
    total_days = len(emotion_df)
    
    print(f"\n统计信息:")
    print(f"总交易日: {total_days}")
    print(f"冰点天数: {ice_days} ({ice_days/total_days*100:.1f}%)")
    print(f"高潮天数: {climax_days} ({climax_days/total_days*100:.1f}%)")
else:
    print("暂无市场情绪数据，请先运行数据采集")

## 4. 龙头识别

In [ ]:
from features.leader_features import LeaderFeatures

# 构造示例数据
dates = pd.date_range('2024-01-01', '2024-12-31', freq='D')

# 模拟龙头股（强势）
leader_data = pd.DataFrame({
    'close': np.random.randn(len(dates)).cumsum() + 100,
    'volume': np.random.randint(5000000, 20000000, len(dates)),
    'amount': np.random.randint(50000000, 200000000, len(dates))
}, index=dates)
leader_data['close'] = leader_data['close'] * 1.5  # 模拟强势

# 模拟跟风股（弱势）
follower_data = pd.DataFrame({
    'close': np.random.randn(len(dates)).cumsum() + 100,
    'volume': np.random.randint(1000000, 5000000, len(dates)),
    'amount': np.random.randint(10000000, 50000000, len(dates))
}, index=dates)

# 基准数据
benchmark_data = pd.DataFrame({
    'close': np.random.randn(len(dates)).cumsum() + 3000,
}, index=dates)

# 计算龙头特征
leader_features = LeaderFeatures.calculate_leader_score(
    symbol='LEADER',
    stock_df=leader_data,
    benchmark_df=benchmark_data,
    start_date='2024-06-01',
    end_date='2024-12-31'
)

follower_features = LeaderFeatures.calculate_leader_score(
    symbol='FOLLOWER',
    stock_df=follower_data,
    benchmark_df=benchmark_data,
    start_date='2024-06-01',
    end_date='2024-12-31'
)

# 对比
comparison = pd.DataFrame({
    '龙头股': leader_features,
    '跟风股': follower_features
})

print("龙头 vs 跟风 特征对比:")
print(comparison)

# 可视化
comparison.T.plot(kind='bar', figsize=(12, 6))
plt.title('龙头股 vs 跟风股 特征对比')
plt.ylabel('特征值')
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 5. 策略回测

In [ ]:
# 这里展示如何加载回测结果
import yaml

results_dir = project_root / 'results'

# 获取最新的回测结果
if results_dir.exists():
    result_files = sorted(results_dir.glob('backtest_*.yaml'), reverse=True)
    
    if result_files:
        latest_result = result_files[0]
        print(f"加载回测结果: {latest_result.name}")
        
        with open(latest_result, 'r', encoding='utf-8') as f:
            results = yaml.safe_load(f)
        
        print("\n策略配置:")
        print(yaml.dump(results['config'], allow_unicode=True))
        
        print("\n回测结果:")
        if 'portfolio_metrics' in results:
            for key, value in results['portfolio_metrics'].items():
                print(f"  {key}: {value}")
    else:
        print("暂无回测结果，请先运行回测")
else:
    print("results/ 目录不存在")

## 6. 结果分析

In [ ]:
# 这里可以添加更详细的结果分析
# 例如：收益曲线、回撤分析、交易统计等

print("TODO: 添加详细的结果分析")
print("- 收益曲线")
print("- 最大回撤分析")
print("- 月度收益分布")
print("- 交易统计（胜率、盈亏比等）")
print("- 持仓分析")

## 7. 参数优化

In [ ]:
# 参数网格搜索示例

param_grid = {
    'topk': [5, 10, 15, 20],
    'max_positions': [2, 3, 5],
    'position_size': [0.2, 0.3, 0.4],
    'ice_point_threshold': [800, 1000, 1200],
}

print("参数优化建议:")
print("1. 使用网格搜索找到最优参数组合")
print("2. 注意过拟合风险")
print("3. 使用交叉验证")
print("4. 在样本外数据上测试")

print("\n参数网格:")
for key, values in param_grid.items():
    print(f"  {key}: {values}")

## 总结

这个 Notebook 展示了如何分析和优化龙头低吸策略。

下一步:
1. 完善回测结果分析
2. 实现参数优化
3. 添加更多特征
4. 对接实盘